# 🎬 SoniSlowVideo — slow VIDEO to fit natural voice

**Different project from SoniTranslate main.**

Repo: [5656wwed/SoniSlowVideo](https://github.com/5656wwed/SoniSlowVideo)

### What this does
- Keeps Kokoro voice **natural** (does not rush speech)
- **Slows the video** so it matches the longer voice track
- SRT mode: uses your subtitle text as-is (no translate)

### How to use
1. Runtime → Change runtime type → **T4 GPU**
2. Run Step 1 (install ~10 min)
3. Run Step 2 (paste HF token, launch)
4. Upload **VIDEO** + **SRT** (advanced settings)
5. Source language = language of your SRT
6. Output type = **video (mp4)**

⚠️ Accept pyannote license: [speaker-diarization](https://hf.co/pyannote/speaker-diarization) + [segmentation](https://hf.co/pyannote/segmentation)  
Token: [hf.co/settings/tokens](https://hf.co/settings/tokens) (Read access to gated repos)

In [ ]:
#@markdown ## Step 1: Install SoniSlowVideo
import os, sys, time

%cd /content
!rm -rf /content/SoniSlowVideo
!git clone -q https://github.com/5656wwed/SoniSlowVideo.git
%cd SoniSlowVideo
!git checkout -q f5-tts 2>/dev/null || echo "f5-tts branch not found, using main"

!pip uninstall chex pandas-stubs ibis-framework albumentations albucore jax numpy -y -q 2>/dev/null
!pip install -q uv==0.8.13
!uv venv --python 3.10 --clear -q
!curl -sS https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!uv run python get-pip.py pip==23.1.2 -q
!uv run python -m pip install -q pip==23.1.2 Setuptools==80.6.0
!apt-get install -qq -y git-lfs ffmpeg 2>/dev/null
!git lfs install

!uv run python -m pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124

!sed -i 's|git+https://github.com/R3gm/whisperX.git@cuda_11_8|git+https://github.com/R3gm/whisperX.git@cuda_12_x|' requirements_base.txt
!uv run python -m pip install -q -r requirements_base.txt
!uv run python -m pip install -q -r requirements_extra.txt
!uv run python -m pip install -q onnxruntime-gpu==1.22.0

!uv run python -m pip install -q piper-tts==1.2.0
!uv run python -m pip install -q -r requirements_xtts.txt 2>/dev/null
!uv run python -m pip install -q TTS==0.21.1 --no-deps 2>/dev/null
!uv run python -m pip install -q kokoro
!uv run python -m pip install -q pocket-tts
!uv run python -m pip install -q f5-tts

!sudo apt-get install -y libcudnn8 -q 2>/dev/null || echo "libcudnn8 skipped"
!uv run python -m pip install -q "numpy<2.0" --force-reinstall --no-deps

print("\n✅ SoniSlowVideo installed. Run Step 2.\n")
time.sleep(1)

In [ ]:
#@markdown ## Step 1.5: Your cloned voices (saved to Drive, reuse forever)
# Your voices persist in Google Drive. First run: upload each voice once.
# Later runs: this cell just reloads them automatically (no re-upload).
import os, shutil

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

VOICE_DIR = '/content/drive/MyDrive/SoniSlow_F5_Voices'
os.makedirs(VOICE_DIR, exist_ok=True)
existing = [f for f in os.listdir(VOICE_DIR) if f.lower().endswith(('.wav','.mp3','.ogg','.m4a','.flac'))]

if not existing:
    # First time: upload your voice sample + type the exact words it says
    from google.colab import files
    print('No voices yet. Upload your voice sample (.wav or .mp3):')
    up = files.upload()
    name = list(up.keys())[0]
    base = os.path.splitext(os.path.basename(name))[0]
    with open(os.path.join(VOICE_DIR, os.path.basename(name)),'wb') as f:
        f.write(up[name])
    ref_text = input('Type the EXACT words your sample says: ').strip()
    with open(os.path.join(VOICE_DIR, base + '.txt'),'w') as f:
        f.write(ref_text)
    print('Saved. New voice ready:', 'en-F5-' + base + ' F5-TTS')
else:
    print('Found your saved voices. No re-upload needed.')

# Link saved voices into the app so they appear in the dropdown
target = '/content/SoniSlowVideo/_F5TTS_'
os.makedirs(target, exist_ok=True)
for fn in os.listdir(VOICE_DIR):
    shutil.copy(os.path.join(VOICE_DIR, fn), os.path.join(target, fn))

print('\nYour voices (in dropdown as en-F5-<name> F5-TTS):')
for f in os.listdir(VOICE_DIR):
    print('  -', f)
print('\nContinue to Step 2.')


In [ ]:
#@markdown ## Step 2: Launch
YOUR_HF_TOKEN = "" #@param {type:'string'}
%env YOUR_HF_TOKEN={YOUR_HF_TOKEN}

theme = "Taithrah/Minimal" # @param ["Taithrah/Minimal", "aliabid94/new-theme", "gstaff/xkcd", "ParityError/LimeFace", "abidlabs/pakistan", "rottenlittlecreature/Moon_Goblin", "ysharma/llamas", "gradio/dracula_revamped"]
interface_language = "english" # @param ['arabic', 'azerbaijani', 'chinese_zh_cn', 'english', 'french', 'german', 'hindi', 'indonesian', 'italian', 'japanese', 'korean', 'marathi', 'polish', 'portuguese', 'russian', 'spanish', 'swedish', 'turkish', 'ukrainian', 'vietnamese']

%cd /content/SoniSlowVideo
!uv run python app_rvc.py --theme {theme} --language {interface_language} --public_url